<a href="https://colab.research.google.com/github/tk-learning-artifacts/ai-accelerator-2026-notebooks/blob/main/student/x-embeddings-1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Beyond the course · Embeddings 1 — What the network sees

In [ ]:
# Installs what Colab doesn't ship with.
!pip install -q medmnist

On Friday you replaced ResNet-18's last layer and trained a new one. But what
goes *into* that last layer? 512 numbers per image, which the network computes
from the picture. These numbers are called an **embedding**, and they are a
summary of the image in the network's own terms. Images the network thinks
are alike get similar numbers.

In this notebook you use those 512 numbers, with no training at all, to build
a search engine for tissue, draw a map of 2,000 histology patches, sort them
into groups without labels, and find patches whose label looks wrong. Every
one of those is a method a capstone team can use.

This is an extension pathway: you work through it alone or with a partner,
at your own pace. You need Friday's fine-tuning notebook behind you. Allow
about two hours.

How it works, as in class: 🔮 predict before you run, ✏️ replace the `...`,
✅ run the check under it. A check that fails tells you what to look at. Each
section ends with a **You're done when** line.

**GPU or not?** Nothing here trains. Section 1 runs ResNet-18 over 2,000
patches once: about a minute on the free CPU, a few seconds on a GPU.

**The downloads.** The notebook does not download anything until you run a
⬇️ download cell. There are two. The first (the second code cell) gets
PathMNIST, the same file as Friday: 206 MB, even though this notebook uses
only a small part of it. It comes from our own fast server, so it usually
takes under a minute. The second, at the start of section 1, gets the
ResNet-18 weights: 45 MB. If a cell stops with an error, run it again. If you
only want to read, skip both.

In [ ]:
import os
import hashlib
import pathlib
import shutil
import urllib.request
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
import torchvision
from torchvision import transforms
import medmnist


def is_done(*values):
    """True once none of the values is still `...`."""
    return all(v is not ... for v in values)


def check(ok, hint):
    if not ok:
        raise AssertionError("Not yet. " + hint)
    print("✅ Correct!")


def fetch_from_mirror(url, dest, sha256):
    """Download url to dest and check its sha256. Returns True if the file is good."""
    # The original data servers get slow when a whole class downloads at once,
    # so we keep a copy of each file on our own fast server.
    dest = pathlib.Path(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)

    def good(path):
        with open(path, "rb") as f:
            return hashlib.file_digest(f, "sha256").hexdigest() == sha256

    if dest.exists() and good(dest):
        return True
    # Download to a .part file first, so a stopped download never looks finished.
    part = dest.with_name(dest.name + ".part")
    try:
        # Our server turns away requests that look like they come from a robot,
        # so we give ours a name.
        request = urllib.request.Request(url, headers={"User-Agent": "tk-accelerator-notebooks"})
        with urllib.request.urlopen(request, timeout=30) as reply, open(part, "wb") as out:
            shutil.copyfileobj(reply, out)
        if not good(part):
            raise ValueError("the checksum does not match")
        part.replace(dest)
        return True
    except Exception:
        part.unlink(missing_ok=True)
        print("Our copy of the data did not download, so we are using the original source.")
        return False


class DataMissing(Exception):
    """Stops a cell with one plain message instead of a long error."""

    def __init__(self, message):
        super().__init__(message)
        self.message = message

    def _render_traceback_(self):
        return [self.message]


def require_data(name, *paths):
    """Stop with a plain message unless the downloaded files are on disk."""
    if not all(pathlib.Path(p).exists() for p in paths):
        raise DataMissing(f"The {name} data isn't downloaded yet. Run the ⬇️ Download cell above first.")


def require_model(name, *paths):
    """Stop with a plain message unless the downloaded model files are on disk."""
    if not all(pathlib.Path(p).exists() for p in paths):
        raise DataMissing(f"The {name} isn't downloaded yet. Run the ⬇️ Download cell above first.")


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# The instructors' checker sets NB_CHECK to run this notebook quickly; ignore it.
QUICK = os.environ.get("NB_CHECK") == "1"
weights = torchvision.models.ResNet18_Weights.DEFAULT
RESNET_FILE = pathlib.Path(torch.hub.get_dir()) / "checkpoints" / os.path.basename(weights.url)
torch.manual_seed(0)
np.random.seed(0)
print("Running on", device)

⬇️ **Download the data (206 MB).** Only run the next cell if you are going to use this data. It downloads 206 MB (the whole PathMNIST file, even though this notebook uses only a small part of it), so skip it if you are only reading. It is safe to run twice. "Run all" runs it too, so use "Run all" only when you mean to do the whole notebook.

In [ ]:
# ⬇️ Download cell: PathMNIST (206 MB)
if pathlib.Path("data/pathmnist.npz").exists():
    print("PathMNIST is already downloaded. Nothing to do.")
else:
    if not fetch_from_mirror("https://data.tk.sg/pathmnist/pathmnist.npz", "data/pathmnist.npz",
                             "81823f52dc622e69db2db4c72f8e8e617938dd6864d3c1f23d4e49724a28ea72"):
        medmnist.PathMNIST(split="val", download=True, root="data")
    print("Done: PathMNIST downloaded (206 MB).")

In [ ]:
os.makedirs("data", exist_ok=True)
SHORT = ["adipose", "background", "debris", "lymphocytes", "mucus",
         "muscle", "normal mucosa", "stroma", "TUMOUR"]
tf = transforms.Compose([transforms.Resize(112), transforms.ToTensor(),
                         transforms.Normalize(weights.transforms().mean, weights.transforms().std)])
require_data("PathMNIST", "data/pathmnist.npz")
patches = medmnist.PathMNIST(split="val", download=False, root="data", transform=tf)
raw = medmnist.PathMNIST(split="val", download=False, root="data")
N = 200 if QUICK else 2000
pick = np.random.permutation(len(patches))[:N]
labels = np.array([int(patches.labels[i][0]) for i in pick])
print(N, "patches; how many of each tissue:", np.bincount(labels, minlength=9))


def show_patches(indices, titles=None, size=1.4):
    fig, axes = plt.subplots(1, len(indices), figsize=(size * len(indices), size * 1.3), squeeze=False)
    for k, (ax, i) in enumerate(zip(axes[0], indices)):
        ax.imshow(raw[int(pick[i])][0])
        ax.set_title(titles[k] if titles else SHORT[labels[i]], fontsize=7)
        ax.axis("off")
    plt.show()

## 1. The 512 numbers before the head

ResNet-18 ends with `model.fc`, a `Linear` layer from 512 numbers to 1,000
ImageNet scores. Replace it with `nn.Identity()`, a layer that passes its
input straight through, and the model's output *is* the 512 numbers.

Here it is for one patch, keeping the full model for comparison:

⬇️ **Download the model (45 MB).** Only run the next cell if you are going to use this model. It downloads the ResNet-18 weights, 45 MB, so skip it if you are only reading. It is safe to run twice. "Run all" runs it too, so use "Run all" only when you mean to do the whole notebook.

In [ ]:
# ⬇️ Download cell: ResNet-18 weights (45 MB)
if RESNET_FILE.exists():
    print("The ResNet-18 weights are already downloaded. Nothing to do.")
else:
    weights.get_state_dict(progress=True)
    print("Done: ResNet-18 weights downloaded (45 MB).")

In [ ]:
require_model("ResNet-18 weights file", RESNET_FILE)
full = torchvision.models.resnet18(weights=weights).eval()
image, _ = patches[int(pick[0])]
with torch.no_grad():
    print("full model output:", tuple(full(image[None]).shape))

🔮 **Predict.** After `model.fc = nn.Identity()`, what shape comes out for a
batch of 2 patches?

In [ ]:
my_prediction = ""  # @param {type:"string"}

✏️ **Exercise 1a.** Make `backbone`: ResNet-18 with `weights`, its `fc`
replaced by `nn.Identity()`, in eval mode (`.eval()`), on `device`.

In [ ]:
backbone = ...

In [ ]:
check(is_done(backbone), "Make backbone.")
with torch.no_grad():
    out = backbone(torch.zeros(2, 3, 112, 112).to(device))
check(tuple(out.shape) == (2, 512), f"Two patches in should give (2, 512) out, not {tuple(out.shape)}. Did you replace fc with nn.Identity()?")
check(not backbone.training, "Call .eval(): batch norm must use its stored statistics, not this batch's.")

Everyone continues from here. This runs the backbone over all the patches,
256 at a time, and keeps the embeddings as one NumPy array.

In [ ]:
#@title Run this to continue (about a minute on CPU) { display-mode: "form" }
require_model("ResNet-18 weights file", RESNET_FILE)
net = torchvision.models.resnet18(weights=weights)
net.fc = nn.Identity()
backbone = net.eval().to(device)

started = time.time()
loader = torch.utils.data.DataLoader(torch.utils.data.Subset(patches, pick.tolist()), batch_size=256)
chunks = []
with torch.no_grad():
    for images, _ in loader:
        chunks.append(backbone(images.to(device)).cpu())
emb = torch.cat(chunks).numpy()
print("embeddings:", emb.shape, f"in {time.time() - started:.0f} s")

**You're done when** you have `emb`, one row of 512 numbers for each patch.

## 2. Nearest neighbours: a search engine for tissue

Two embeddings are alike if they point the same way. **Cosine similarity**
measures that: scale each embedding to length 1, then take the dot product.
1 means the same direction, 0 means unrelated.

ResNet-18's 512 numbers are never negative (a ReLU just before them turns
negatives into 0), so here the cosine is always between 0 and 1, and even
two very different patches often score around 0.5. Compare similarities with
each other, not with 0.

In [ ]:
unit = emb / np.linalg.norm(emb, axis=1, keepdims=True)    # every row now has length 1
similar = unit @ unit[0]                                     # patch 0 against every patch
print("patch 0 against itself:", round(float(similar[0]), 3))
print("most and least similar to patch 0:", round(float(np.sort(similar)[-2]), 3), round(float(similar.min()), 3))

✏️ **Exercise 2a.** Write `nearest(i, k)`: the indices of the `k` patches
most similar to patch `i`, most similar first, **not including `i` itself**.
`np.argsort(x)` gives the indices that sort `x` from smallest to largest;
`[::-1]` reverses a list.

In [ ]:
def nearest(i, k=8):
    similar = unit @ unit[i]
    return ...

In [ ]:
got = nearest(0, 5)
check(is_done(got), "nearest still returns `...`.")
check(len(got) == 5, f"Ask for 5, get 5: you returned {len(got)}.")
check(0 not in list(got), "Leave out the patch itself: it is always its own best match.")
sims = unit @ unit[0]
check(np.all(np.diff(sims[got]) <= 1e-6), "Most similar first: the similarities should go down along the list.")
check(np.isclose(sims[got[0]], np.sort(sims)[-2]), "The first neighbour should be the most similar patch other than itself.")

Everyone continues from here.

In [ ]:
#@title Run this to continue { display-mode: "form" }
def nearest(i, k=8):
    ranked = np.argsort(-(unit @ unit[i]))
    return ranked[ranked != i][:k]

Pick a patch and see its neighbours. Change `query` and run again.

In [ ]:
query = 0  # @param {type:"integer"}
near = nearest(query)
show_patches([query] + list(near), titles=["QUERY\n" + SHORT[labels[query]]] + [SHORT[labels[j]] for j in near])

🔮 **Predict.** If you guess each patch's tissue as the most common label among
its 5 nearest neighbours, with no training at all, what fraction will you get
right? Nine tissues, so guessing at random gives about 11%.

In [ ]:
my_prediction = ""  # @param {type:"string"}

In [ ]:
guesses = np.array([np.bincount(labels[nearest(i, 5)], minlength=9).argmax() for i in range(N)])
print(f"5 nearest neighbours, no training: {np.mean(guesses == labels):.0%} right")

**You're done when** you have seen the neighbours of at least three patches
and know how well "ask the neighbours" classifies.

## 3. A map of 2,000 patches

512 numbers can't be drawn. **t-SNE** (short for "t-distributed stochastic
neighbour embedding"; you don't need the name) squeezes them down to 2 numbers per patch
so that patches that were close neighbours stay close on the page. It is a map
of who-is-next-to-whom: distances *within* a group mean something, but the
gaps *between* groups and the groups' sizes mostly do not.

First **PCA** (principal component analysis) shrinks 512 numbers to 50
while keeping most of the variation (it is faster, and t-SNE works better on
it), then t-SNE draws the map. `perplexity=30` tells t-SNE roughly how many
neighbours each patch should try to keep close to it: about 30. Try 5 and 100
later and see how the map changes.

🔮 **Predict.** Which tissues will sit close together on the map, and which
will be off on their own? Write down two of each.

In [ ]:
my_prediction = ""  # @param {type:"string"}

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

started = time.time()
flat = PCA(n_components=50, random_state=0).fit_transform(emb)
xy = TSNE(n_components=2, perplexity=30, random_state=0, init="pca").fit_transform(flat)
print(f"map drawn in {time.time() - started:.0f} s")

plt.figure(figsize=(8, 7))
for k in range(9):
    here = labels == k
    plt.scatter(xy[here, 0], xy[here, 1], s=6, label=SHORT[k])
plt.legend(markerscale=3, fontsize=8)
plt.title("2,000 tissue patches, placed by ResNet-18's embeddings")
plt.axis("off")
plt.show()

🤔 **Think.** The network was trained on photos of dogs, cars and food, and has
never seen a tissue label. Why do the tissues still form groups?

**You're done when** you have the map and have checked your predictions
against it.

## 4. Groups without labels

What if there were no labels at all? **k-means** sorts points into `k` groups
(clusters): it places `k` centres, gives every point to its nearest centre,
moves each centre to the middle of its points, and repeats. It never sees a
label.

In [ ]:
from sklearn.cluster import KMeans

clusters = KMeans(n_clusters=9, n_init=10, random_state=0).fit_predict(unit)
table = np.zeros((9, 9), dtype=int)
for c, t in zip(clusters, labels):
    table[c, t] += 1
print("rows: clusters 0–8; columns:", [s[:6] for s in SHORT])
print(table)

Each row is a cluster, and each column a true tissue. A good cluster has most
of its patches in one column.

✏️ **Exercise 4a.** Write `purity(table)`: for each cluster (row), take its
biggest count, add them up, and divide by the total number of patches. 1
means every cluster holds one tissue only.

In [ ]:
def purity(table):
    return ...

In [ ]:
demo = np.array([[8, 2, 0],
                 [1, 1, 8],
                 [0, 5, 5]])
check(is_done(purity(demo)), "purity still returns `...`.")
check(np.isclose(purity(demo), 21 / 30), f"For the demo: (8 + 8 + 5) / 30 = 0.7, not {purity(demo)}. Take each row's biggest number.")
check(np.isclose(purity(np.eye(3, dtype=int) * 4), 1.0), "Clusters that each hold one tissue should score 1.")

In [ ]:
print(f"purity of 9 clusters with no labels: {purity(table):.2f}")
for c in range(9):
    print(f"cluster {c}: mostly {SHORT[table[c].argmax()]} ({table[c].max()} of {table[c].sum()})")

**You're done when** you have a purity and can name the tissue that k-means
splits or mixes most.

## 5. Patches whose label looks wrong

People label data, and people make mistakes. A patch whose nearest neighbours
almost all carry a *different* label is either a labelling mistake or an
unusual case. Both are worth a look before you train on the data.

🔮 **Predict.** Of the 2,000 patches, how many will have at least 9 of their 10
nearest neighbours labelled differently from themselves?

In [ ]:
my_prediction = ""  # @param {type:"string"}

In [ ]:
disagree = np.array([np.sum(labels[nearest(i, 10)] != labels[i]) for i in range(N)])
odd = np.where(disagree >= 9)[0]
print(len(odd), "patches disagree with at least 9 of their 10 neighbours")
show = odd[np.argsort(-disagree[odd])][:8]
neighbour_vote = [SHORT[np.bincount(labels[nearest(i, 10)], minlength=9).argmax()] for i in show]
if len(show):
    show_patches(list(show), titles=[f"label: {SHORT[labels[i]]}\nneighbours: {v}" for i, v in zip(show, neighbour_vote)], size=1.8)

🤔 **Discuss.** For each patch shown, would you believe the label or the
neighbours? What would a team do with such patches in Project I: delete
them, relabel them, or keep them and say so?

**You're done when** you have looked at the odd patches and decided, for at
least two, whether you trust the label.

### Stretch: make it yours

- **Duplicates across splits.** Load 2,000 patches from `split="train"` too,
  embed them, and find train–validation pairs with cosine similarity above
  0.99. Near-copies in both splits make validation scores too optimistic
  (Wednesday's leakage).
- **Fine-tuned embeddings.** Fine-tune the backbone on PathMNIST as on Friday,
  then redraw the map. Do the groups get tighter?
- **Project II, stage 3.** This one needs Cellpose, which this notebook
  does not install, so do it in your copy of **Day 4 · Notebook 8 — Project
  II starter: finding and counting cells** (`d4-segmentation-starter.ipynb`).
  Copy this notebook's `backbone` and embedding code across. Cut a small
  square around each nucleus that Cellpose found, embed each square, and
  cluster into 2 groups. Do the groups match brown-stained (positive) and
  blue-only (negative) cells?
- **Fool the model.** This one needs Friday's trained tissue model, so do it
  in your copy of **Day 4 · Notebook 7 — Fine-tuning a pretrained model, and
  Grad-CAM** (`d4-finetune.ipynb`). Take one patch as a tensor `image` and
  call `image.requires_grad_(True)`, so PyTorch keeps a slope for every
  pixel. Run the model, call `.backward()` on the wrong class's score, and
  `image.grad` now says which way each pixel would raise that score. Add a
  tiny step, `0.01 * image.grad.sign()`, and see how small a change flips the
  model's answer.

In [ ]:
# Your stretch work here.